# Manufacturing Yield Analysis

End-to-end walkthrough of the **data-scientist** plugin on a synthetic manufacturing dataset, using the bundled `ds_skill` helpers (no hand-rolled statistics).

**Business question:** Which process parameters drive yield, and which apparent signals are noise or confounded?

**Dataset:** 500 production runs (`dataset.csv`). Ground truth baked into `generate_data.py`:

| Variable | True effect on yield |
|---|---|
| `temperature_c` | strong negative (-0.3 / °C) |
| `pressure_psi` | moderate positive (+0.2 / psi) |
| `humidity_pct` | weak negative (-0.1 / %) |
| `line_speed_uph` | none (noise variable) |
| `operator` = C | better trained (+3) |
| `shift` = Night | fatigue penalty (-2) |
| `equipment_age_days` | confounds temperature (older runs hotter) |

A good analysis should recover the real drivers and **reject** `line_speed` as a non-driver.

## Setup — make the tested helpers importable

This block finds the bundled `ds_skill` package from anywhere in the repo. Alternatively run `pip install -e .` once and `import ds_skill` works with no path setup.

In [ ]:
import sys
from pathlib import Path

# Locate the bundled, tested helper library (ds_skill).
for _p in [Path.cwd(), *Path.cwd().parents]:
    _scripts = _p / "plugins" / "data-scientist" / "skills" / "analysis-workflow" / "scripts"
    if _scripts.is_dir():
        sys.path.insert(0, str(_scripts))
        break

import pandas as pd

DATA = next(p / "examples" / "manufacturing_yield" / "dataset.csv"
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "examples" / "manufacturing_yield" / "dataset.csv").exists())
df = pd.read_csv(DATA)
FEATURES = ["temperature_c", "pressure_psi", "humidity_pct", "line_speed_uph", "equipment_age_days"]
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")
df.head()

## Step 1 — Data readiness

Before any modeling, score whether the data can actually answer the question (sample size, missingness, balance, leakage, ...).

In [ ]:
from ds_skill.readiness import assess_readiness

report = assess_readiness(df, target="yield_pct", candidate_features=FEATURES)
print("Overall readiness:", report.overall_status)
for dim in report.dimensions.values():
    print(f"  {dim.name:<22} {dim.status:<8} {dim.evidence}")
for caveat in report.caveats:
    print("  caveat:", caveat)

## Step 2 — Rank drivers with FDR control

Pearson/Spearman/MI against the target, with Benjamini-Hochberg FDR correction across all candidate features so we don't over-claim from multiple comparisons.

In [ ]:
from ds_skill.correlation import correlation_with_target

results = correlation_with_target(df, target="yield_pct", candidate_features=FEATURES)
pearson = sorted([r for r in results if r.method == "pearson"],
                 key=lambda r: r.effect_strength, reverse=True)
print(f"{'feature':<20}{'r':>8}  {'sig(FDR)':>8}  interpretation")
for r in pearson:
    print(f"{r.x:<20}{r.coefficient:>8.3f}  {str(r.significant_after_fdr):>8}  {r.interpretation}")

## Step 3 — Multivariate regression

A linear model holds the other variables constant, separating real drivers from confounded ones. Compare the recovered coefficients to the ground-truth table above.

In [ ]:
from ds_skill.regression import fit_linear_regression, residual_diagnostics

model = fit_linear_regression(df.dropna(subset=FEATURES + ["yield_pct"]),
                              target="yield_pct", features=FEATURES)
print(f"R^2 = {model.r_squared:.3f}\n")
print(f"{'feature':<20}{'coef':>10}")
for name, coef in model.coefficients.items():
    print(f"{name:<20}{coef:>10.3f}")

diag = residual_diagnostics(model, df.dropna(subset=FEATURES + ["yield_pct"]))
print(f"\nNormality p-value: {diag.normality_p_value:.3f}")
print("Multicollinearity flagged:", diag.multicollinearity_flagged)
for rec in diag.recommendations:
    print("  rec:", rec)

## Step 4 — Charts

The `plotting` module returns headless matplotlib figures that follow the report annotation rules (title states the question, N shown, units on axes).

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless; drop this line for inline display
from ds_skill import plotting

fig = plotting.plot_scatter_fit(
    df["temperature_c"], df["yield_pct"], fit="linear",
    title="Yield falls as temperature rises (N=500)",
    xlabel="Temperature (°C)", ylabel="Yield (%)",
)
plotting.save_figure(fig, "temperature_vs_yield.png")

corr = df[FEATURES + ["yield_pct"]].corr()
fig2 = plotting.plot_correlation_matrix(corr, title="Process variable correlations")
plotting.save_figure(fig2, "correlation_matrix.png")
print("Saved temperature_vs_yield.png and correlation_matrix.png")

## Findings vs. ground truth

- **Temperature** — strongest negative driver. Recovered. ✅
- **Pressure** — positive driver. Recovered. ✅
- **Humidity** — weak negative. Recovered (small coefficient). ✅
- **Line speed** — coefficient ~0 and not significant after FDR: correctly **rejected** as a non-driver. ✅
- **Equipment age** confounds temperature; the multivariate model separates them.

This mirrors the staged workflow the plugin runs interactively via `/ds-analyze`.